In [2]:
import cv2
import pytesseract
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from PIL import Image
import torch
import numpy as np
from huggingface_hub import snapshot_download

local_path = "./trocr_handwritten"


# Load pre-trained TrOCR model
processor = TrOCRProcessor.from_pretrained(local_path)
model = VisionEncoderDecoderModel.from_pretrained(local_path)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Function: Segment lines using OpenCV
def segment_lines(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    _, binary = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # Dilation to merge characters into lines
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (30, 20))
    dilated = cv2.dilate(binary, kernel, iterations=1)

    # Find contours for each line
    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    line_images = []
    boxes = []

    for cnt in sorted(contours, key=lambda x: cv2.boundingRect(x)[1]):
        x, y, w, h = cv2.boundingRect(cnt)
        line_img = img[y:y+h, x:x+w]
        line_images.append(line_img)
        boxes.append((x, y, w, h))
    
    return line_images, boxes

# Function: Recognize text using TrOCR
def recognize_line_images(line_images):
    results = []
    for line in line_images:
        pil_img = Image.fromarray(line)
        if pil_img.mode != "RGB":
            pil_img = pil_img.convert("RGB")
        pixel_values = processor(images=pil_img, return_tensors="pt").pixel_values.to(device)
        generated_ids = model.generate(pixel_values)
        text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        results.append(text)
    return results

# Main pipeline
def process_paragraph_image(image_path):
    line_images, boxes = segment_lines(image_path)
    texts = recognize_line_images(line_images)
    
    # Output text with coordinates
    for i, (text, (x, y, w, h)) in enumerate(zip(texts, boxes)):
        print(f"Line {i+1}: {text}")
        print(f"Bounding Box: x={x}, y={y}, w={w}, h={h}")
        print("")

# Example usage
image_path = "himage3.jpeg"
process_paragraph_image(image_path)

Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at ./trocr_handwritten and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Line 1: " Mr. Powell finds it easier to take it out of mothers ,
Bounding Box: x=57, y=53, w=1123, h=107

Line 2: children and sick people than to take on this vast
Bounding Box: x=63, y=173, w=1126, h=104

Line 3: " Mr. Brown commented icily . " Let us have a
Bounding Box: x=262, y=287, w=935, h=108

Line 4: industrie ,
Bounding Box: x=61, y=287, w=209, h=93

